In [1]:
!pip3 install scikit-learn


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import pandas as pd
import numpy as np
import ast
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [ ]:
df = pd.read_csv("../DataSets/MovieReviews_Lemmatized.csv")

def join_tokens(text):
    if pd.isna(text):
        return ""
    try:
        tokens = ast.literal_eval(text)
        return " ".join(tokens)
    except:
        return text

df["Cleaned_Text"] = df["Reviews"].apply(join_tokens)

df = df.dropna(subset=['Cleaned_Text', 'emotion'])

print("Veri boyutu:", df.shape)
df[['Cleaned_Text', 'emotion']].head(3)

Veri boyutu: (19316, 5)


,Cleaned_Text,emotion
0,laugh overall motivation character incomprehen...,anticipation
1,wait exhale wait wait wait wait get point wait...,anticipation
2,angela basset good expect whitney range actres...,anticipation


In [ ]:
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))

X = tfidf.fit_transform(df["Cleaned_Text"])
y = df["emotion"]

X_train, X_test, y_train, y_test, indices_train, indices_test = train_test_split(
    X, y, df.index, test_size=0.2, random_state=42
)

print(f"Eğitim seti boyutu: {X_train.shape}")
print(f"Test seti boyutu: {X_test.shape}")

Eğitim seti boyutu: (15452, 10000)
Test seti boyutu: (3864, 10000)


In [ ]:
models = {
    "SVM (Linear)": LinearSVC(C=0.5, class_weight='balanced', random_state=42, dual=False),
    "Logistic Regression": LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42),
    "Naive Bayes": MultinomialNB(alpha=0.5),
    "Random Forest": RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1)
}

results = []
best_model_name = ""
best_accuracy = 0
best_predictions = None

for model_name, model in models.items():
    print(f"--- Eğitiliyor: {model_name} ---")
    
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    
    results.append({
        "Model": model_name, 
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1
    })
    
    print(f"Accuracy: {acc:.4f} | F1-Score: {f1:.4f}\n")
    
    if acc > best_accuracy:
        best_accuracy = acc
        best_model_name = model_name
        best_predictions = y_pred

results_df = pd.DataFrame(results).sort_values(by="F1-Score", ascending=False)
print("=== DETAYLI MODEL KARŞILAŞTIRMA SONUÇLARI ===")
print(results_df)

results_df.to_csv("../Results/Model_Comparison_Results-2.csv", index=False)
print("\nDetaylı model karşılaştırma sonuçları 'Model_Comparison_Results.csv' olarak kaydedildi.")

test_results_df = df.loc[indices_test].copy()
test_results_df["Predicted_Emotion"] = best_predictions
test_results_df["Used_Model"] = best_model_name
output_df = test_results_df[["Index", "movie_name", "Cleaned_Text", "emotion", "Predicted_Emotion", "Used_Model"]]
output_df.to_csv("../Results/Test_Predictions-2.csv", index=False)

--- Eğitiliyor: SVM (Linear) ---
Accuracy: 0.5065 | F1-Score: 0.5121

--- Eğitiliyor: Logistic Regression ---
Accuracy: 0.4532 | F1-Score: 0.4630

--- Eğitiliyor: Naive Bayes ---
Accuracy: 0.4764 | F1-Score: 0.3694

--- Eğitiliyor: Random Forest ---
Accuracy: 0.4552 | F1-Score: 0.3361

=== DETAYLI MODEL KARŞILAŞTIRMA SONUÇLARI ===
                 Model  Accuracy  Precision    Recall  F1-Score
0         SVM (Linear)  0.506470   0.524328  0.506470  0.512146
1  Logistic Regression  0.453157   0.509056  0.453157  0.463043
2          Naive Bayes  0.476449   0.648288  0.476449  0.369366
3        Random Forest  0.455228   0.607253  0.455228  0.336075

Detaylı model karşılaştırma sonuçları 'Model_Comparison_Results.csv' olarak kaydedildi.
